# Dia 3 — Limpeza e Preparação de Dados
Dataset: Diamantes | Objetivo: aprender a identificar e corrigir problemas nos dados antes de qualquer análise

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import shutil
import os

In [5]:
# Baixa e copia o dataset (só precisa rodar uma vez)
path = kagglehub.dataset_download("shivam2503/diamonds")
shutil.copy(f"{path}/diamonds.csv", "../data/diamonds.csv")

# Carrega o dataset
df = pd.read_csv('../data/diamonds.csv')

# Primeira olhada
print("Shape:", df.shape)
print("\nPrimeiras linhas:")
df.head()

Shape: (53940, 11)

Primeiras linhas:


,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


## Diagnóstico do Dataset
Antes de limpar, precisamos entender o que temos — tipos de dados, valores ausentes e estatísticas gerais.

In [6]:
# Tipos de dados e valores ausentes
print("=== Informações gerais ===")
df.info()

print("\n=== Valores ausentes por coluna ===")
print(df.isnull().sum())

=== Informações gerais ===
<class 'pandas.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53940 non-null  int64  
 1   carat       53940 non-null  float64
 2   cut         53940 non-null  str    
 3   color       53940 non-null  str    
 4   clarity     53940 non-null  str    
 5   depth       53940 non-null  float64
 6   table       53940 non-null  float64
 7   price       53940 non-null  int64  
 8   x           53940 non-null  float64
 9   y           53940 non-null  float64
 10  z           53940 non-null  float64
dtypes: float64(6), int64(2), str(3)
memory usage: 5.1 MB

=== Valores ausentes por coluna ===
Unnamed: 0    0
carat         0
cut           0
color         0
clarity       0
depth         0
table         0
price         0
x             0
y             0
z             0
dtype: int64


## Limpeza dos Dados

### Problemas identificados:
1. Coluna `Unnamed: 0` é um índice duplicado — será removida
2. Colunas `cut`, `color` e `clarity` devem ser do tipo `category`
3. Colunas `x`, `y`, `z` podem conter zeros — fisicamente impossível num diamante

In [8]:
# Remove coluna desnecessária
df = df.drop(columns=['Unnamed: 0'])

# Verifica zeros nas dimenções físicas
print("=== Zeros nas dimensões físicas ===")
print("x == 0:", (df['x'] == 0).sum())
print("y == 0:", (df['y'] == 0).sum())
print("z == 0:", (df['z'] == 0).sum())


=== Zeros nas dimensões físicas ===
x == 0: 8
y == 0: 7
z == 0: 20


## Tratamento de Outliers e Erros
Dimensões físicas iguais a zero são impossíveis — indicam erro de registro.
Esses registros serão removidos do dataset.

In [9]:
# Registros antes da limpeza
print("Registros antes:", len(df))

# Remove linhas onde x, y ou z são zero
df = df[(df['x'] > 0) & (df['y'] > 0) & (df['z'] > 0)]

# Registros depois da limpeza
print("Registros depois:", len(df))
print("Registros removidos:", 53940 - len(df))


Registros antes: 53940
Registros depois: 53920
Registros removidos: 20


## Conversão de Tipos de Dados
As colunas `cut`, `color` e `clarity` são categorias com ordem lógica.
Convertê-las para o tipo `category` melhora performance e abre possibilidades de análise ordenada.

In [10]:
# Define a ordem lógica de cada categoria
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Converte para category ordenada
df['cut'] = pd.Categorical(df['cut'], categories=cut_order, ordered=True)
df['color'] = pd.Categorical(df['color'], categories=color_order, ordered=True)
df['clarity'] = pd.Categorical(df['clarity'], categories=clarity_order, ordered=True)

# Confirma
print(df[['cut', 'color', 'clarity']].dtypes)
print("\nExemplo — ordem do corte:")
print(df['cut'].cat.categories)

cut        category
color      category
clarity    category
dtype: object

Exemplo — ordem do corte:
Index(['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'], dtype='str')


## Estatísticas Descritivas
Com o dataset limpo, vamos explorar os números gerais — médias, mínimos, máximos e distribuição dos dados.

In [12]:
# Estatísticas das colunas numéricas
df.describe()

,carat,depth,table,price,x,y,z
count,53920.000000,53920.000000,53920.000000,53920.000000,53920.000000,53920.000000,53920.000000
mean,0.797698,61.749514,57.456834,3930.993231,5.731627,5.734887,3.540046
std,0.473795,1.432331,2.234064,3987.280446,1.119423,1.140126,0.702530
min,0.200000,43.000000,43.000000,326.000000,3.730000,3.680000,1.070000
25%,0.400000,61.000000,56.000000,949.000000,4.710000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5323.250000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


## Outliers nas Dimensões
O `describe()` revelou valores improváveis em `y` e `z`.
Vamos investigar e remover esses registros.

In [14]:
# Investiga os outliers
print("=== Valores extremos em y ===")
print(df[df['y'] >20][['carat', 'price', 'x', 'y', 'z']])

print("\n=== Valores extremos em z ===")
print(df[df['z'] > 20][['carat', 'price', 'x', 'y', 'z']])

=== Valores extremos em y ===
       carat  price     x     y     z
24067   2.00  12210  8.09  58.9  8.06
49189   0.51   2075  5.15  31.8  5.12

=== Valores extremos em z ===
       carat  price     x     y     z
48410   0.51   1970  5.12  5.15  31.8


## Remoção de Outliers Improváveis
Três registros apresentam valores inconsistentes em `y` e `z` quando comparados com `x`.
Provável erro de digitação — serão removidos.

In [15]:
# Remove outliers improváveis
print("Registros antes:", len(df))

df = df[(df['y'] < 20) & (df['z'] < 20)]

print("Registros depois:", len(df))
print("Registros removidos:", 53920 - len(df))

Registros antes: 53920
Registros depois: 53917
Registros removidos: 3


## Resumo da Limpeza
Vamos documentar tudo que foi feito no dataset.

In [16]:
print("=== Resumo da Limpeza ===")
print(f"Registros originais:  53.940")
print(f"Registros removidos:")
print(f"  - Dimensões zeradas (x, y ou z = 0): 20")
print(f"  - Outliers improváveis (y ou z > 20): 3")
print(f"  - Total removido:                    23")
print(f"Registros finais:     {len(df)}")
print(f"\nColunas removidas:")
print(f"  - Unnamed: 0 (índice duplicado)")
print(f"\nTipos corrigidos:")
print(f"  - cut, color, clarity → category ordenada")

=== Resumo da Limpeza ===
Registros originais:  53.940
Registros removidos:
  - Dimensões zeradas (x, y ou z = 0): 20
  - Outliers improváveis (y ou z > 20): 3
  - Total removido:                    23
Registros finais:     53917

Colunas removidas:
  - Unnamed: 0 (índice duplicado)

Tipos corrigidos:
  - cut, color, clarity → category ordenada


## Validação Final
Conferindo as estatísticas após a limpeza para garantir que os dados fazem sentido.

In [17]:
# Estatísticas finais
print("=== Estatísticas finais ===")
print(df.describe().round(2))

print("\n=== Distribuição dos cortes ===")
print(df['cut'].value_counts())

=== Estatísticas finais ===
          carat     depth     table     price         x         y         z
count  53917.00  53917.00  53917.00  53917.00  53917.00  53917.00  53917.00
mean       0.80     61.75     57.46   3930.91      5.73      5.73      3.54
std        0.47      1.43      2.23   3987.22      1.12      1.11      0.69
min        0.20     43.00     43.00    326.00      3.73      3.68      1.07
25%        0.40     61.00     56.00    949.00      4.71      4.72      2.91
50%        0.70     61.80     57.00   2401.00      5.70      5.71      3.53
75%        1.04     62.50     59.00   5323.00      6.54      6.54      4.04
max        5.01     79.00     95.00  18823.00     10.74     10.54      6.98

=== Distribuição dos cortes ===
cut
Ideal        21547
Premium      13779
Very Good    12080
Good          4902
Fair          1609
Name: count, dtype: int64


## Salvando o Dataset Limpo
Exportando o dataset tratado para uso nos próximos dias.

In [18]:
# Salva o dataset limpo
df.to_csv('../data/diamonds_clean.csv', index=False)
print("Dataset salvo em ../data/diamonds_clean.csv")
print(f"Shape final: {df.shape}")

Dataset salvo em ../data/diamonds_clean.csv
Shape final: (53917, 10)
